# Time-of-day sensitivity: app–vote correlations per window

Loads the 10 per-window sRCA files from notebook 8.1 (5 windows × 2 elections), rebuilds the urban+suburban modelling table per window, saves a CSV per window for the downstream Dirichlet re-fit, and plots how the app–party correlations change across time windows and years. Addresses Reviewer 76A (sensitivity to the hour choice).

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

import os
WORKING_DIR = os.environ.get("REPO_ROOT", os.path.abspath(".."))  # repo root (notebooks run from notebooks/)
DATA_DIR = f'{WORKING_DIR}/data'
IMG_DIR = f'{WORKING_DIR}/images'
os.makedirs(f'{DATA_DIR}/dirichlet', exist_ok=True)

## Configuration

In [ ]:
# Windows produced by 8.1 (suffix -> label)
WINDOWS = ['20_07', '07_20', '07_13', '13_20', 'full']
WINDOW_LABELS = {'20_07': '20-07 (home)', '07_20': '07-20 (day)', '07_13': '07-13 (morning)',
                 '13_20': '13-20 (afternoon)', 'full': 'full day'}
ELECTIONS = ['europe_2019', 'europe_2024']

# sRCA columns by category (same set as notebook 9)
social_media = ['Facebook_srca', 'Instagram_srca', 'LinkedIn_srca', 'SnapChat_srca',
                'Twitter_srca', 'Twitch_srca', 'TikTok_srca']
news = ['NewsPaper_srca', 'Sports News_srca', 'DailyMotion_srca', 'NewsMag_srca',
        'Google News_srca', 'TV5MONDE_srca']
messaging = ['WhatsApp_srca', 'Apple iMessage_srca', 'Signal_srca', 'Discord_srca', 'Telegram_srca']
streamming = ['Youtube_srca', 'Spotify_srca', 'CanalPlus_srca', 'Netflix_srca', 'Apple Music_srca',
              'Disney+_srca', 'Apple Video_srca', 'Molotov TV_srca', 'Pluto TV_srca']
app_srca_cols = social_media + news + messaging + streamming

demographics = ['median_income', 'unemployment_ratio']
rows = demographics + app_srca_cols          # correlation-matrix rows (top to bottom)
socio_econ_cols = ['median_income', 'unemployment_ratio', 'pop_0_14', 'pop_15_29',
                   'pop_30_44', 'pop_45_59', 'pop_60_74', 'pop_75_89', 'pop_90']

def get_simplify_type(level):
    if level.startswith('rural'): return 'rural'
    if level == 'urbain dense': return 'urban'
    if level == 'urbain densité intermédiaire': return 'suburban'
    return 'NA'

def standarize_variable(values):
    # divide by the (population) standard deviation; Pearson r is scale-invariant,
    # so this only matters for the saved CSVs feeding the Dirichlet model.
    values = np.array(values, dtype=float)
    scale = np.sqrt(np.mean((values - values.mean()) ** 2))
    return values / scale if scale > 0 else values

## Load base data (elections, socioeconomic, urbanization)

In [ ]:
elections = pickle.load(open(f'{DATA_DIR}/elections/elections_communes.pkl', 'rb'))

socio_by_year = {
    2019: pd.read_pickle(f'{DATA_DIR}/dossier/df_social_economical_2019.pkl'),
    2024: pd.read_pickle(f'{DATA_DIR}/dossier/df_social_economical_2024.pkl'),
}

# urbanization level from the carto CSV (same mapping as notebook 7)
df_urb = pd.read_csv(f'{DATA_DIR}/carto/communes_urban_rural.csv')
df_urb['insee'] = df_urb['insee'].astype(str).str.zfill(5)
df_urb['urbanization_level'] = df_urb['type'].apply(get_simplify_type)
df_urb = df_urb[['insee', 'urbanization_level']]
print('urbanization levels:', df_urb['urbanization_level'].value_counts().to_dict())

## Build per-window datasets (urban + suburban) and save CSVs

In [ ]:
def build_window_df(election, window):
    year = elections[election]['date'].year
    res = elections[election]['results'].copy()
    socio = socio_by_year[year].copy()
    socio = socio[socio['pop'] > 0]
    traffic = pd.read_pickle(
        f'{DATA_DIR}/traffic/df_traffic_commune_apps_with_rca_{election}_{window}.pkl')

    for d in (res, socio, traffic):
        d['insee'] = d['insee'].astype(str).str.zfill(5)

    srca_cols = [c for c in app_srca_cols if c in traffic.columns]
    traffic = traffic[['insee'] + srca_cols]

    df = (res.merge(socio, on='insee', how='inner')
             .merge(df_urb, on='insee', how='inner')
             .merge(traffic, on='insee', how='inner')
             .dropna())

    # standardize socio + sRCA on the full merged set (mirrors notebook 9)
    for col in socio_econ_cols + srca_cols:
        df[col] = standarize_variable(df[col])

    # keep urban + suburban (paper geography)
    df_urban = df[df['urbanization_level'] != 'rural'].copy()
    return df_urban, srca_cols


data_by = {}    # (election, window) -> (df_urban, srca_cols, main_parties)
for election in ELECTIONS:
    res_cols = list(elections[election]['results'].columns)
    main_parties = [c for c in res_cols if c.endswith('_votes') and c != 'others_votes']
    for window in WINDOWS:
        df_urban, srca_cols = build_window_df(election, window)
        data_by[(election, window)] = (df_urban, srca_cols, main_parties)
        keep = (['insee'] + main_parties + ['others_votes', 'polarization_dalton', 'ideology']
                + socio_econ_cols + srca_cols)
        keep = [c for c in keep if c in df_urban.columns]
        out = f'{DATA_DIR}/dirichlet/df_data_{election}_{window}.csv'
        df_urban[keep].to_csv(out, index=False)
        print(f'{election} {window}: {df_urban.shape[0]} urban communes, '
              f'{len(srca_cols)} apps, {len(main_parties)} parties -> {out}')

## App–party correlation matrices per window

In [ ]:
def get_correlation_matrix(df, rows, parties):
    C = np.zeros((len(rows), len(parties)))
    P = np.zeros_like(C)                       # 1 = significant (p<0.05), else 0
    for i, r in enumerate(rows):
        present = (r in df.columns) and (np.std(df[r].values) > 0)
        for j, p in enumerate(parties):
            if present:
                c, pv = pearsonr(df[r].values, df[p].values)
            else:
                c, pv = 0.0, 1.0
            C[i, j] = c
            P[i, j] = 1.0 if pv < 0.05 else 0.0
    return C, P


corr_by = {}
for (election, window), (df_urban, srca_cols, main_parties) in data_by.items():
    C, P = get_correlation_matrix(df_urban, rows, main_parties)
    corr_by[(election, window)] = (C, P, main_parties)
print('computed correlation matrices for', len(corr_by), 'window-year combinations')

## Plots: correlation strength + per-window heatmaps

In [ ]:
# Punchline: mean |significant correlation| over the APP rows (excludes demographics),
# per window and year. If the residential window (20-07) is highest or comparable, the
# hour choice is justified / results are robust to it.
app_row_idx = [i for i, r in enumerate(rows) if r in app_srca_cols]

summary = {}
for (election, window), (C, P, parties) in corr_by.items():
    sig = P[app_row_idx, :] > 0
    vals = np.abs(C[app_row_idx, :])[sig]
    summary[(election, window)] = vals.mean() if sig.any() else 0.0

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(WINDOWS)); bw = 0.38
for k, election in enumerate(ELECTIONS):
    yv = [summary[(election, w)] for w in WINDOWS]
    bars = ax.bar(x + (k - 0.5) * bw, yv, bw, label=election.replace('europe_', ''))
    for xi, v in zip(x + (k - 0.5) * bw, yv):
        ax.text(xi, v, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels([WINDOW_LABELS[w] for w in WINDOWS], rotation=20, ha='right')
ax.set_ylabel('mean |significant rho| (app rows)')
ax.set_title('App-party correlation strength by time window')
ax.legend(); plt.tight_layout()
plt.savefig(f'{IMG_DIR}/sensitivity_corr_strength.pdf', bbox_inches='tight')
plt.show()

print(pd.DataFrame({el: [summary[(el, w)] for w in WINDOWS] for el in ELECTIONS},
                   index=WINDOWS).round(4))

In [ ]:
# Detailed view: app-party correlation heatmap per window (one figure per year).
# Non-significant cells (p>=0.05) are left blank.
for election in ELECTIONS:
    parties = corr_by[(election, WINDOWS[0])][2]
    party_labels = [p.replace('_votes', '') for p in parties]
    fig, axes = plt.subplots(1, len(WINDOWS), figsize=(3.4 * len(WINDOWS), 9), sharey=True)
    for ax, window in zip(axes, WINDOWS):
        C, P, _ = corr_by[(election, window)]
        M = np.where(P > 0, C, np.nan)
        im = ax.imshow(M, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
        ax.set_title(WINDOW_LABELS[window], fontsize=10)
        ax.set_xticks(range(len(parties)))
        ax.set_xticklabels(party_labels, rotation=90, fontsize=7)
        ax.set_yticks(range(len(rows)))
    axes[0].set_yticklabels([r.replace('_srca', '') for r in rows], fontsize=7)
    fig.colorbar(im, ax=axes, fraction=0.015, pad=0.01)
    fig.suptitle(f'App-party correlations by window — {election}', y=1.01)
    plt.savefig(f'{IMG_DIR}/sensitivity_corr_heatmaps_{election}.pdf', bbox_inches='tight')
    plt.show()